# LFP dataset: Voltage and SOC ranges of different tail settings

Adapted from the supplied dataset-visualization workflow. This notebook uses the
existing MAT files in `../../dataset`; no duplicate dataset copy is needed.
Figures: `dataset_visualization/outputs/tail-ranges`. Numerical source tables and
input hashes: `dataset_visualization/results/tail-ranges`.
Run all cells in order. See the parent README for definitions and interpretation.


In [ ]:
from pathlib import Path
import sys

# Resolve from a notebook subfolder, the LFP module, or the release root.
roots = (Path.cwd(), *Path.cwd().parents)
candidates = [p for root in roots for p in
              (root, root / "dataset_visualization", root / "LFP dataset/dataset_visualization")]
PROJECT_ROOT = next(p for p in candidates if (p / "notebook_setup.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from notebook_setup import get_dataset_path, save_figure, save_table, finish_category, begin_category
CATEGORY = 'tail-ranges'
begin_category(CATEGORY)


import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import os

# ============================================================
# 0. Global Nature-style settings
# ============================================================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ============================================================
# 1. Load data
# ============================================================
mat_path = get_dataset_path("LFP_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat = data["processed_Qd_mat"].astype(float)
processed_Vd_mat = data["processed_Vd_mat"].astype(float)

n_rows, n_cells = processed_Qd_mat.shape
Qmax_all = processed_Qd_mat[-1, :].ravel()

# ============================================================
# 2. Basic settings
# ============================================================
V_full_low = 2.50
V_full_high = 3.65
V_full_range = V_full_high - V_full_low

Vtail_upp_list = [2.73, 2.75, 2.77]

save_dir = "Tail_voltage_SOC_ratio_barplots"
# The shared exporter creates the output directory.

# Nature-like muted colours
voltage_color = "#4C72B0"     # voltage proportion
soc_color = "#DD8452"         # SOC proportion
unused_color = "#B8B8B8"      # remaining proportion

# ============================================================
# 3. Helper function: calculate SOC at a given voltage
# ============================================================
def calculate_soc_at_voltage(Vd_i, Qd_i, Qmax_i, target_voltage):
    """
    Calculate SOC at a given discharge voltage by interpolation.

    SOC = Q(target_voltage) / Qmax
    """
    valid_mask = np.isfinite(Vd_i) & np.isfinite(Qd_i)

    if not np.any(valid_mask):
        return np.nan

    if not np.isfinite(Qmax_i) or Qmax_i <= 0:
        return np.nan

    Vd_valid = Vd_i[valid_mask]
    Qd_valid = Qd_i[valid_mask]

    v_min = np.nanmin(Vd_valid)
    v_max = np.nanmax(Vd_valid)

    if not (v_min <= target_voltage <= v_max):
        return np.nan

    sort_idx = np.argsort(Vd_valid)
    Vd_sorted = Vd_valid[sort_idx]
    Qd_sorted = Qd_valid[sort_idx]

    Q_target = np.interp(target_voltage, Vd_sorted, Qd_sorted)
    SOC_target = Q_target / Qmax_i

    return SOC_target


# ============================================================
# 4. Plot tail voltage ratio + SOC at tail upper cut-off voltage
# ============================================================
for V_tail_upp in Vtail_upp_list:

    # ------------------------------------------------------------
    # 4.1 Voltage proportion of the tail segment
    # ------------------------------------------------------------
    tail_voltage_ratio = (V_tail_upp - V_full_low) / V_full_range * 100.0

    # ------------------------------------------------------------
    # 4.2 SOC corresponding to the tail upper cut-off voltage
    # ------------------------------------------------------------
    SOC_tail_upp = np.full(n_cells, np.nan, dtype=float)

    for i in range(n_cells):
        SOC_tail_upp[i] = calculate_soc_at_voltage(
            Vd_i=processed_Vd_mat[:, i],
            Qd_i=processed_Qd_mat[:, i],
            Qmax_i=Qmax_all[i],
            target_voltage=V_tail_upp
        )

    SOC_tail_upp_percent = SOC_tail_upp * 100.0
    mean_SOC_tail_upp_percent = np.nanmean(SOC_tail_upp_percent)
    save_table(CATEGORY, f'cell_SOC_Vtail_{V_tail_upp:.2f}.csv', cell_index_zero_based=np.arange(n_cells), tail_cutoff_SOC_percent=SOC_tail_upp_percent)
    save_table(CATEGORY, f'proportions_Vtail_{V_tail_upp:.2f}.csv', measure=['Voltage', 'SOC'], percent=[tail_voltage_ratio, mean_SOC_tail_upp_percent])

    # ------------------------------------------------------------
    # 4.3 Values for plotting
    # ------------------------------------------------------------
    used_values = [
        tail_voltage_ratio,
        mean_SOC_tail_upp_percent
    ]

    unused_values = [
        100.0 - tail_voltage_ratio,
        100.0 - mean_SOC_tail_upp_percent
    ]

    x = np.arange(2)

    # ------------------------------------------------------------
    # 4.4 Create figure
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(2.2, 2))

    # Voltage proportion bar: tail region
    ax.bar(
        x[0],
        used_values[0],
        width=0.65,
        color=voltage_color,
        alpha=0.95,
        edgecolor="black",
        linewidth=0.6,
        label="Tail region"
    )

    # Voltage proportion bar: remaining region, no legend
    ax.bar(
        x[0],
        unused_values[0],
        width=0.65,
        bottom=used_values[0],
        color=unused_color,
        alpha=0.35,
        edgecolor="black",
        linewidth=0.6,
        label="_nolegend_"
    )

    # SOC proportion bar: tail region, no duplicate legend
    ax.bar(
        x[1],
        used_values[1],
        width=0.65,
        color=soc_color,
        alpha=0.95,
        edgecolor="black",
        linewidth=0.6,
        label="_nolegend_"
    )

    # SOC proportion bar: remaining region, no legend
    ax.bar(
        x[1],
        unused_values[1],
        width=0.65,
        bottom=used_values[1],
        color=unused_color,
        alpha=0.35,
        edgecolor="black",
        linewidth=0.6,
        label="_nolegend_"
    )

    # ------------------------------------------------------------
    # 4.5 Add value annotations
    # ------------------------------------------------------------
    for xi, value in zip(x, used_values):
        ax.text(
            xi,
            value + 3,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=12
        )

    # ------------------------------------------------------------
    # 4.6 Axis formatting
    # ------------------------------------------------------------
    ax.set_ylim(0, 100)
    ax.set_xlim(-0.6, 1.6)

    ax.set_ylabel("Proportion (%)", labelpad=2)

    ax.set_xticks(x)
    ax.set_xticklabels(
        ["Voltage", "SOC"],
        fontsize=12,
    )

    ax.set_yticks([0, 50, 100])

    ax.grid(False)
    ax.set_title("")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(axis="both", labelsize=12)

    # ------------------------------------------------------------
    # 4.7 Legend: only show Tail region
    # ------------------------------------------------------------
    ax.legend(
        frameon=False,
        fontsize=12,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.32),
        ncol=1,
        handlelength=1.2,
        handletextpad=0.4
    )

    plt.tight_layout(pad=0.4)

    # ------------------------------------------------------------
    # 4.8 Save PNG
    # ------------------------------------------------------------
    save_name = f"Tail_voltage_SOC_cutoff_ratio_Vtail_{V_tail_upp:.2f}.png"
    save_path = os.path.join(save_dir, save_name)

    save_figure(fig, CATEGORY, 
        save_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(
        f"Saved: {save_path} | "
        f"Voltage ratio = {tail_voltage_ratio:.1f}% | "
        f"Mean SOC at tail cut-off = {mean_SOC_tail_upp_percent:.1f}%"
    )

print("All figures have been generated.")
finish_category(CATEGORY)
